# M1 SAR Ship Detection — Kaggle 2×T4 Benchmark

**Dataset:** HRSID — 4,042 train / 1,962 val · 800×800 SAR JPEG chips · single class: ship  
**Models:** YOLOv8m (2023 baseline) → YOLO11m-OBB (2024 oriented boxes) → YOLO26m (2026 Ultralytics flagship)  
**Hardware:** Kaggle 2×T4 (30 GB VRAM total), DDP, ~2 h/model → ~6 h total  
**Paper story:** horizontal-box CNN → tight oriented-box CNN → NMS-free 2026 SOTA

### Before running
1. Enable GPU: Settings → Accelerator → **2× NVIDIA T4 GPU**  
2. Enable internet: Settings → **Internet on**  
3. Add secrets in Kaggle (Settings → Secrets):
   - `HF_TOKEN` — your HuggingFace WRITE token  
   - `WANDB_API_KEY` — your Weights & Biases API key  

Expected HRSID mAP50 from literature: YOLOv8m ~88-90% · YOLO11m-OBB ~91-93% · YOLO26m ~93-94%

## 0 — Setup

In [ ]:
%%capture
# Upgrade ultralytics to get YOLO26 support
!pip install -q -U ultralytics huggingface_hub wandb gdown
import ultralytics; print(f'ultralytics {ultralytics.__version__}')

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
from collections import defaultdict

WORK = Path('/kaggle/working')
REPO = WORK / 'internship'

# Clone the project repo
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/shaunmarv3/internship.git', str(REPO)],
        check=True
    )
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(f'Repo: {REPO}  |  cwd: {os.getcwd()}')

In [ ]:
# Auth — read from Kaggle secrets
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN']       = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY']  = secrets.get_secret('WANDB_API_KEY')
    print('Secrets loaded: HF_TOKEN and WANDB_API_KEY')
except Exception as e:
    print(f'WARNING: Could not load secrets ({e}).\n'
          'Set HF_TOKEN / WANDB_API_KEY manually in the next cell if needed.')

In [ ]:
# Uncomment to set manually if secrets aren't available
# os.environ['HF_TOKEN']      = 'hf_xxx'
# os.environ['WANDB_API_KEY'] = 'xxx'

# W&B login (non-interactive — uses the env var set above)
import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY'), relogin=True)
print('W&B ready')

## 1 — Download HRSID

In [ ]:
# HRSID main (614 MB) + pure-background negatives (400 images)
RAW = WORK / 'hrsid_raw'
RAW.mkdir(exist_ok=True)

MAIN_ZIP  = RAW / 'hrsid.zip'
NEG_ZIP   = RAW / 'hrsid_neg.zip'

if not MAIN_ZIP.exists():
    print('Downloading HRSID main (~614 MB)...')
    import gdown
    gdown.download(id='1NY3ovgc-woDlNoQdyqzRB3t9McOBH5Ms', output=str(MAIN_ZIP), quiet=False)
else:
    print(f'Already downloaded: {MAIN_ZIP}')

if not NEG_ZIP.exists():
    print('Downloading HRSID negatives (~220 MB)...')
    import gdown
    gdown.download(id='1U0Sj1SHoq-2VjXXUKwpXae6rBI3YjyDP', output=str(NEG_ZIP), quiet=False)
else:
    print(f'Already downloaded: {NEG_ZIP}')

print('Downloads complete.')

In [ ]:
import zipfile

EXTRACT = RAW / 'extracted'
EXTRACT.mkdir(exist_ok=True)

if not any(EXTRACT.rglob('*.jpg')):
    print('Extracting main zip...')
    with zipfile.ZipFile(MAIN_ZIP) as zf:
        zf.extractall(EXTRACT)
    print('Main zip extracted.')

NEG_EXTRACT = RAW / 'negatives'
NEG_EXTRACT.mkdir(exist_ok=True)
if not any(NEG_EXTRACT.rglob('*.png')):
    print('Extracting negatives zip...')
    with zipfile.ZipFile(NEG_ZIP) as zf:
        zf.extractall(NEG_EXTRACT)
    print('Negatives extracted.')

# Locate key files
train_json = next(EXTRACT.rglob('train2017.json'))
val_json   = next(EXTRACT.rglob('test2017.json'))
img_root   = next(EXTRACT.rglob('*.jpg')).parent
neg_pngs   = sorted(NEG_EXTRACT.rglob('*.png'))

print(f'Train JSON : {train_json}')
print(f'Val JSON   : {val_json}')
print(f'Image root : {img_root}  ({len(list(img_root.glob("*.jpg")))} jpgs)')
print(f'Negatives  : {len(neg_pngs)} png files')

## 2 — COCO → YOLO Conversion (horizontal boxes)

Creates `HRSID_yolo/images/train|val` and `HRSID_yolo/labels/train|val`.

In [ ]:
def coco_to_yolo(coco_json_path, src_img_dir, dst_img_dir, dst_lbl_dir):
    """COCO bbox → YOLO txt (normalized cx cy w h). Copies images too."""
    with open(coco_json_path) as f:
        coco = json.load(f)

    dst_img_dir = Path(dst_img_dir); dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir = Path(dst_lbl_dir); dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    src_img_dir = Path(src_img_dir)

    id2img = {img['id']: img for img in coco['images']}
    ann_map = defaultdict(list)
    for ann in coco['annotations']:
        ann_map[ann['image_id']].append(ann)

    copied, labelled = 0, 0
    for img_id, info in id2img.items():
        W, H = info['width'], info['height']
        stem = Path(info['file_name']).stem
        src  = src_img_dir / info['file_name']

        # copy image
        dst_img = dst_img_dir / info['file_name']
        if not dst_img.exists() and src.exists():
            shutil.copy2(src, dst_img)
            copied += 1

        # write label
        lines = []
        for ann in ann_map[img_id]:
            x, y, w, h = ann['bbox']   # COCO: top-left pixel + width/height
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            nw, nh = w / W, h / H
            lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        (dst_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines))
        labelled += 1

    print(f'  images copied: {copied}  labels written: {labelled}')


YOLO_DIR = WORK / 'HRSID_yolo'

print('Converting train split...')
coco_to_yolo(train_json, img_root,
             YOLO_DIR / 'images' / 'train',
             YOLO_DIR / 'labels' / 'train')

print('Converting val split...')
coco_to_yolo(val_json, img_root,
             YOLO_DIR / 'images' / 'val',
             YOLO_DIR / 'labels' / 'val')

print('Done.')

In [ ]:
# Add 400 pure-background negatives to train
# Negatives: .png files with NO bounding boxes → empty label file

NEG_IMG_DIR = YOLO_DIR / 'images' / 'train'
NEG_LBL_DIR = YOLO_DIR / 'labels' / 'train'

# One stem collision documented in PROJECT_PLAN:
# P0128_600_1400_4800_5600.png collides with a train image → rename
COLLISION_STEM = 'P0128_600_1400_4800_5600'

added = 0
for png in neg_pngs:
    stem = png.stem
    if stem == COLLISION_STEM:
        stem = stem + '_neg'
    dst = NEG_IMG_DIR / (stem + '.png')
    if not dst.exists():
        shutil.copy2(png, dst)
    # empty label file = background (no boxes)
    lbl = NEG_LBL_DIR / (stem + '.txt')
    if not lbl.exists():
        lbl.write_text('')
    added += 1

total_train = len(list((YOLO_DIR / 'images' / 'train').glob('*')))
total_val   = len(list((YOLO_DIR / 'images' / 'val').glob('*')))
print(f'Added {added} negatives.  Total train: {total_train}  val: {total_val}')
# Expected: train 4042 (3642 ship + 400 neg), val 1962

In [ ]:
# Write data.yaml for horizontal detection (YOLOv8m and YOLO26m)
yaml_content = f"""path: {YOLO_DIR}
train: images/train
val:   images/val
nc: 1
names: ['ship']
"""
(YOLO_DIR / 'data.yaml').write_text(yaml_content)
print(f'data.yaml written to {YOLO_DIR / "data.yaml"}')
print(yaml_content)

## 3 — OBB Label Conversion (for YOLO11m-OBB)

Fits minimum enclosing rotated rectangle to each COCO polygon annotation.  
Output: 8-corner format `class x1 y1 x2 y2 x3 y3 x4 y4` (normalised).

In [ ]:
import cv2
import numpy as np

def convert_to_obb(coco_json_path, out_lbl_dir):
    """COCO polygon → Ultralytics OBB 4-corner format (normalised)."""
    with open(coco_json_path) as f:
        coco = json.load(f)

    out_lbl_dir = Path(out_lbl_dir)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    id2img = {img['id']: img for img in coco['images']}
    ann_map = defaultdict(list)
    for ann in coco['annotations']:
        if ann.get('segmentation'):
            ann_map[ann['image_id']].append(ann)

    for img_id, info in id2img.items():
        W, H = info['width'], info['height']
        stem = Path(info['file_name']).stem
        lines = []
        for ann in ann_map[img_id]:
            for seg in ann['segmentation']:
                if len(seg) < 6:
                    continue
                pts = np.array(seg, dtype=np.float32).reshape(-1, 2)
                rect = cv2.minAreaRect(pts)   # ((cx,cy),(w,h),angle)
                corners = cv2.boxPoints(rect)  # 4×2 pixel coords
                norm = corners / np.array([W, H])
                line = '0 ' + ' '.join(f'{v:.6f}' for v in norm.flatten())
                lines.append(line)
        (out_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines))

    print(f'  OBB labels → {out_lbl_dir}  ({len(id2img)} files)')


OBB_DIR = WORK / 'HRSID_obb'

# OBB shares the same images as HRSID_yolo — symlink to save disk
(OBB_DIR / 'images').mkdir(parents=True, exist_ok=True)
for split in ['train', 'val']:
    src = YOLO_DIR / 'images' / split
    lnk = OBB_DIR / 'images' / split
    if not lnk.exists():
        lnk.symlink_to(src.resolve())

print('Converting OBB train labels...')
convert_to_obb(train_json, OBB_DIR / 'labels' / 'train')
print('Converting OBB val labels...')
convert_to_obb(val_json,   OBB_DIR / 'labels' / 'val')

# OBB negatives → same empty label files
obb_neg_lbl = OBB_DIR / 'labels' / 'train'
for png in neg_pngs:
    stem = png.stem + ('_neg' if png.stem == COLLISION_STEM else '')
    lbl = obb_neg_lbl / (stem + '.txt')
    if not lbl.exists():
        lbl.write_text('')

# Write OBB data.yaml (task: obb)
obb_yaml = f"""path: {OBB_DIR}
train: images/train
val:   images/val
nc: 1
names: ['ship']
"""
(OBB_DIR / 'data.yaml').write_text(obb_yaml)
print(f'OBB data.yaml written to {OBB_DIR / "data.yaml"}')

## 4 — Verify Dataset

In [ ]:
def count_split(base, split):
    imgs = list((Path(base) / 'images' / split).glob('*'))
    lbls = list((Path(base) / 'labels' / split).glob('*.txt'))
    return len(imgs), len(lbls)

print('YOLO dataset:')
for s in ['train', 'val']:
    n_img, n_lbl = count_split(YOLO_DIR, s)
    print(f'  {s}: {n_img} images, {n_lbl} labels')

print('OBB dataset:')
for s in ['train', 'val']:
    n_img, n_lbl = count_split(OBB_DIR, s)
    print(f'  {s}: {n_img} images, {n_lbl} labels')

# Quick spot check — show 1 label from each
sample_yolo = next((YOLO_DIR / 'labels' / 'train').glob('*.txt'))
sample_obb  = next((OBB_DIR  / 'labels' / 'train').glob('*.txt'))
print(f'\nSample YOLO label ({sample_yolo.name}):')
print(sample_yolo.read_text().splitlines()[:2])
print(f'\nSample OBB label ({sample_obb.name}):')
print(sample_obb.read_text().splitlines()[:2])

In [ ]:
# Visual sanity check — display busiest scene with boxes
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

with open(train_json) as f:
    coco = json.load(f)
id2img = {img['id']: img for img in coco['images']}
ann_map2 = defaultdict(list)
for a in coco['annotations']: ann_map2[a['image_id']].append(a)
busiest_id = max(ann_map2, key=lambda i: len(ann_map2[i]))
binfo = id2img[busiest_id]

img_path = YOLO_DIR / 'images' / 'train' / binfo['file_name']
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(Image.open(img_path).convert('L'), cmap='gray')
for ann in ann_map2[busiest_id]:
    x, y, w, h = ann['bbox']
    ax.add_patch(mpatches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', lw=1.5))
ax.set_title(f"{binfo['file_name']} — {len(ann_map2[busiest_id])} ships")
ax.axis('off'); plt.tight_layout(); plt.show()

## 5 — Training

All models: 50 epochs · 800px · batch=16 (8 per GPU) · patience=15  
DDP activated via `device='0,1'` — Ultralytics handles it automatically.  
Estimated time per model: **~1.5–2 h** on 2×T4 → total ~5–6 h.

### 5a — YOLOv8m (baseline, horizontal boxes)

In [ ]:
os.chdir(REPO)   # ensure we run from repo root so imports resolve

!python src/models/train_detection.py \
    --model    yolov8m \
    --data     {YOLO_DIR}/data.yaml \
    --project  {WORK}/checkpoints/vessel \
    --device   0,1 \
    --push_hf \
    --wandb_project maritime-vessel

### 5b — YOLO11m-OBB (2024 oriented bounding boxes)

Ships are elongated at arbitrary angles in SAR. OBB fits tightly — ~40% less background speckle inside the box vs a horizontal box. Uses HRSID polygon annotations.

In [ ]:
os.chdir(REPO)

!python src/models/train_detection.py \
    --model    yolo11m-obb \
    --data     {OBB_DIR}/data.yaml \
    --project  {WORK}/checkpoints/vessel \
    --device   0,1 \
    --push_hf \
    --wandb_project maritime-vessel

### 5c — YOLO26m (Ultralytics 2026 flagship)

NMS-free, progressive loss, STAL label assignment. Replaces RT-DETR-L as the third benchmark model — same Ultralytics pip package, no custom install needed.

In [ ]:
os.chdir(REPO)

!python src/models/train_detection.py \
    --model    yolo26m \
    --data     {YOLO_DIR}/data.yaml \
    --project  {WORK}/checkpoints/vessel \
    --device   0,1 \
    --push_hf \
    --wandb_project maritime-vessel

## 6 — Results Summary

In [ ]:
import pandas as pd

ckpt_base = WORK / 'checkpoints' / 'vessel'
models = [
    ('YOLOv8m (baseline)',     'hrsid_yolov8m'),
    ('YOLO11m-OBB (oriented)', 'hrsid_yolo11m_obb'),
    ('YOLO26m (2026 flagship)','hrsid_yolo26m'),
]

rows = []
for label, run_name in models:
    csv_path = ckpt_base / run_name / 'results.csv'
    if not csv_path.exists():
        rows.append({'Model': label, 'mAP50': 'pending', 'mAP50-95': 'pending',
                     'Precision': 'pending', 'Recall': 'pending'})
        continue
    df = pd.read_csv(csv_path).iloc[-1]   # last epoch (best val)
    rows.append({
        'Model':      label,
        'mAP50':      f"{df.get('metrics/mAP50(B)', df.get('metrics/mAP50', 'N/A')):.4f}",
        'mAP50-95':   f"{df.get('metrics/mAP50-95(B)', df.get('metrics/mAP50-95', 'N/A')):.4f}",
        'Precision':  f"{df.get('metrics/precision(B)', df.get('metrics/precision', 'N/A')):.4f}",
        'Recall':     f"{df.get('metrics/recall(B)', df.get('metrics/recall', 'N/A')):.4f}",
    })

results_df = pd.DataFrame(rows)
print('\n=== M1 SAR Ship Detection — Final Results ===\n')
print(results_df.to_string(index=False))

In [ ]:
# Save results table to repo for research.md
import json as _json
results_path = REPO / 'checkpoints' / 'vessel' / 'test_comparison.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_json(str(results_path), orient='records', indent=2)
print(f'Saved → {results_path}')

## 7 — Manual HF Push (if --push_hf failed)

Run this cell only if the auto-push during training failed.

In [ ]:
# Uncomment and run if needed
# os.chdir(REPO)
# for run_name in ['hrsid_yolov8m', 'hrsid_yolo11m_obb', 'hrsid_yolo26m']:
#     run_dir = WORK / 'checkpoints' / 'vessel' / run_name
#     best_pt = run_dir / 'weights' / 'best.pt'
#     if best_pt.exists():
#         !python src/models/push_to_hf.py {best_pt} --subfolder vessel/{run_name}
#         print(f'Pushed {run_name}')
#     else:
#         print(f'WARNING: {best_pt} not found')

---
## Notes for research.md

**Model choice rationale (2026-06-24 literature sweep):**
- RT-DETR-L replaced by **YOLO26m**: Ultralytics' official 2026 flagship — no NMS, progressive loss,  
  STAL label assignment, 40.9-57.5 mAP on COCO, ships in `pip install -U ultralytics`.
- **YOLOv12** (NeurIPS 2025): excluded — community repo only (github.com/sunsmarterjie/yolov12),  
  not in official Ultralytics, requires separate conda env + flash-attn.
- **SARES-DEIM** (arXiv 2604.04127, April 2026): SOTA on HRSID at 93.8% mAP50 — paper only,  
  no code released (same situation as OilSAM2 in M3). Cannot reproduce.
- **AC-YOLO** (+1.5% over YOLO11): paper only, custom YOLO11 modification, no pip package.

**HRSID dataset verified facts (from PROJECT_PLAN):**  
- 5,604 images total · 4,042 train (3,642 ship + 400 background neg) · 1,962 val  
- COCO JSON → YOLO horizontal · COCO polygon → OBB (cv2.minAreaRect)  
- Ships = bright point targets on dark ocean; polarisation-robust (all pols)  
- 1 stem collision fixed: `P0128_600_1400_4800_5600` → `P0128_600_1400_4800_5600_neg`